# Follow up Principal-Angle Experiments



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils.extmath import randomized_svd
from tqdm.auto import tqdm
from google.colab import drive

drive.mount("/content/gdrive")

ROOT = Path("/content/gdrive/MyDrive/gemma3_1b_it_gsm8k_raw26_v3_lean")
META = ROOT / "analysis_cache_spectral/rollout_metadata.parquet"
RAW = ROOT / "full/raw_layer_outputs"
OUT = ROOT / "principal_angle_repeated_disjoint"

OUT.mkdir(parents=True, exist_ok=True)

RANK = 5
SEED = 42
EPS = 1e-12

EXPERIMENTS = {
    1000: 9,
    500: 10,
    2000: 4,
    625: 15,
}


## Load eligible rollout population

In [ ]:
df = pd.read_parquet(META)

df = df[
    df["correct"].notna() &
    df["boxed_token_start"].notna()
].copy()

df["hidden_path"] = df.apply(
    lambda r: str(RAW / f"{r.sample_id}_r{int(r.rollout_id):02d}.npz"),
    axis=1
)

df = df[df["hidden_path"].map(lambda p: Path(p).exists())].copy()

df["sample_id"] = df["sample_id"].astype(str)
df["rollout_id"] = df["rollout_id"].astype(str)
df["key"] = list(zip(df["sample_id"], df["rollout_id"]))

print("Eligible rollouts:", len(df))

for n, n_iter in EXPERIMENTS.items():
    needed = n * n_iter
    print(f"N={n}: {n_iter} iterations need {needed} rollouts")
    assert needed <= len(df), f"N={n}: not enough rollouts"


## Principal-angle pipeline

In [ ]:
def load_repr(row):
    with np.load(row.hidden_path) as z:
        H = z["causal_hidden_states"][
            :, :int(row.boxed_token_start), :
        ].astype(np.float32)

    H /= np.maximum(
        np.linalg.norm(H, axis=-1, keepdims=True),
        EPS
    )
    return H.mean(axis=1)


def principal_angles(sample, desc=""):
    X = np.stack([
        load_repr(r)
        for _, r in tqdm(
            sample.iterrows(),
            total=len(sample),
            desc=f"{desc} | loading",
            leave=False
        )
    ])

    y = sample["correct"].astype(bool).to_numpy()
    assert y.sum() >= RANK and (~y).sum() >= RANK

    rows = []

    for layer in tqdm(
        range(X.shape[1]),
        desc=f"{desc} | layers",
        leave=False
    ):
        C = X[y, layer].copy()
        I = X[~y, layer].copy()

        C -= C.mean(axis=0, keepdims=True)
        I -= I.mean(axis=0, keepdims=True)

        _, _, Vc = randomized_svd(
            C, n_components=RANK, random_state=SEED
        )
        _, _, Vi = randomized_svd(
            I, n_components=RANK, random_state=SEED
        )

        s = np.linalg.svd(Vc @ Vi.T, compute_uv=False)
        theta = np.degrees(np.arccos(np.clip(s, 0.0, 1.0)))

        for k, angle in enumerate(theta, 1):
            rows.append((layer, k, angle))

    return pd.DataFrame(
        rows,
        columns=["layer", "angle", "degrees"]
    )


## Run one experiment size

In [ ]:
def run_experiment(n, n_iter):
    exp_dir = OUT / f"N_{n}"
    exp_dir.mkdir(parents=True, exist_ok=True)

    pool = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    all_results = []

    for i in tqdm(
        range(n_iter),
        desc=f"N={n} iterations",
        unit="iteration"
    ):
        iteration = i + 1
        iter_dir = exp_dir / f"iteration_{iteration:02d}"
        iter_dir.mkdir(parents=True, exist_ok=True)

        start = i * n
        stop = start + n
        sample = pool.iloc[start:stop].copy()

        sample.to_csv(
            iter_dir / "sampled_rollouts.csv",
            index=False
        )

        y = sample["correct"].astype(bool)
        print(
            f"N={n} | iteration={iteration:02d} | "
            f"correct={y.sum()} | incorrect={(~y).sum()}"
        )

        result = principal_angles(
            sample,
            desc=f"N={n} iter={iteration:02d}"
        )
        result["n"] = n
        result["iteration"] = iteration

        result.to_csv(
            iter_dir / "principal_angles.csv",
            index=False
        )

        fig, axes = plt.subplots(
            1, RANK,
            figsize=(18, 3.5),
            sharey=True
        )

        for k, ax in enumerate(axes, 1):
            q = result[result["angle"] == k]
            ax.plot(q["layer"], q["degrees"])
            ax.set_title(f"θ{k}")
            ax.set_xlabel("Layer")
            ax.grid(alpha=0.2)

        axes[0].set_ylabel("Principal angle (°)")
        fig.suptitle(f"N={n} | Iteration {iteration:02d}")
        plt.tight_layout()

        fig.savefig(
            iter_dir / "principal_angles.png",
            dpi=200,
            bbox_inches="tight"
        )
        plt.close(fig)

        all_results.append(result)

    all_results = pd.concat(all_results, ignore_index=True)

    all_results.to_csv(
        exp_dir / "all_iterations.csv",
        index=False
    )

    seen = set()
    for i in range(n_iter):
        sample = pool.iloc[i*n:(i+1)*n]
        keys = set(sample["key"])
        assert not seen & keys
        seen |= keys

    print(
        f"N={n}: COMPLETE | "
        f"{len(seen)} unique rollouts across {n_iter} iterations"
    )

    return all_results


## Experiment 1 — N = 500

In [ ]:
results_500 = run_experiment(
    n=500,
    n_iter=EXPERIMENTS[500]
)



## Experiment 2 — N = 625

In [ ]:
results_625 = run_experiment(
    n=625,
    n_iter=EXPERIMENTS[625]
)


## Experiment 3 — N = 1000


In [ ]:
results_1000 = run_experiment(
    n=1000,
    n_iter=EXPERIMENTS[1000]
)


## Experiment 4 - N = 2000

In [ ]:
results_2000 = run_experiment(
    n=2000,
    n_iter=EXPERIMENTS[2000]
)

## Summary plots across iterations

In [ ]:
for n, results in [
    (1000, results_1000),
    (500, results_500),
    (2000, results_2000),
    (625,results_625)
]:
    fig, axes = plt.subplots(
        1, RANK,
        figsize=(18, 3.5),
        sharey=True
    )

    for k, ax in enumerate(axes, 1):
        q = results[results["angle"] == k]

        for iteration in sorted(q["iteration"].unique()):
            x = q[q["iteration"] == iteration]
            ax.plot(
                x["layer"],
                x["degrees"],
                alpha=0.65,
                label=f"Iter {iteration:02d}"
            )

        ax.set_title(f"θ{k}")
        ax.set_xlabel("Layer")
        ax.grid(alpha=0.2)

    axes[0].set_ylabel("Principal angle (°)")
    axes[-1].legend(fontsize=7)
    fig.suptitle(f"All disjoint iterations | N={n}")
    plt.tight_layout()

    exp_dir = OUT / f"N_{n}"
    fig.savefig(
        exp_dir / "all_iterations_overlay.png",
        dpi=200,
        bbox_inches="tight"
    )
    plt.show()


## Output structure

```text
principal_angle_repeated_disjoint/
├── N_1000/
│   ├── iteration_01/
│   │   ├── sampled_rollouts.csv
│   │   ├── principal_angles.csv
│   │   └── principal_angles.png
│   ├── ...
│   ├── all_iterations.csv
│   ├── all_iterations_overlay.png
│  
├── N_500/
└── N_200/
```
